# Task 3 — Combine & Improve: Hybrid Retrieval with Groundedness Verification
Dataset: TriviaQA (`rc.nocontext`) — Project 10

## 0. Overview

Fuse Task 1's BM25 ranking + Task 2's dense ranking via RRF (k=60) -> generate with the hybrid
top-5 (Colab GPU) -> groundedness check (Task 1's lexical-overlap idea). **Headline Task 3 result
is the pass-1 hybrid-retrieval + generation system** (Section 7). Section 9 additionally tests
whether retrying ungrounded answers with expanded top-10 context improves things further -- it
does not (net negative, see Section 9/10); it is reported as a diagnostic experiment, not used
for the headline numbers. Run on Colab (GPU) for generation; locally reuses a prior Colab run's
results otherwise (same pattern as Task 2).

## 1. Setup

In [1]:
# Importing the libraries we need for this task
import json
import re
import string
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Fixing the seed everywhere so results stay reproducible
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Detecting whether we're on Colab (GPU available) or running locally
# (local runs skip generation and just reuse a prior Colab run's output file)
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running on Colab:" if IN_COLAB else "Running locally:", IN_COLAB)

Running locally: False


In [2]:
# Installing the Colab-only dependencies (locally we assume requirements.txt is already set up)
if IN_COLAB:
    get_ipython().system('pip install -q rank_bm25 sentence-transformers faiss-cpu nltk accelerate "bitsandbytes>=0.46.1"')
    print("Installed Colab-only deps. If bitsandbytes was just installed for the first time, "
          "Runtime > Restart session, then re-run from this cell.")
else:
    print("Local run: assuming `pip install -r ../requirements.txt` has already been run.")

Local run: assuming `pip install -r ../requirements.txt` has already been run.


In [3]:
# Setting up the config constants used throughout this notebook
K_LIST = [1, 3, 5, 10]
TOP_K = 5
TOP_K_RETRIEVE = 10
RRF_K = 60  # spec: RRF constant = 60
MAX_PASSAGE_TOKENS = 200
MAX_NEW_TOKENS = 64
TEMPERATURE = 0.0
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
GROUNDEDNESS_THRESHOLD = 0.5  # fraction of the answer's content words that must appear
                               # in the retrieved evidence to call it "grounded"

# Setting up the filenames for the shared sample/index built in Task 1
SAMPLE_FILENAME = "triviaqa_sample_300.jsonl"
INDEX_FILENAME = "wiki_paragraph_index.jsonl"
GENERATION_RESULTS_FILENAME = "task3_generation_results.jsonl"

# On Colab we need the sample + paragraph index uploaded by hand; locally we just
# read them straight from the data/ folder built by Task 1
if IN_COLAB:
    from google.colab import files
    print(f"Upload {SAMPLE_FILENAME} and {INDEX_FILENAME} (from this repo's data/ folder):")
    uploaded = files.upload()
    assert SAMPLE_FILENAME in uploaded and INDEX_FILENAME in uploaded, (
        "Both files are required -- re-run this cell and upload them."
    )
    SAMPLE_PATH = Path(SAMPLE_FILENAME)
    INDEX_PATH = Path(INDEX_FILENAME)
    RESULTS_DIR = Path(".")
else:
    DATA_DIR = Path("../../23cs30011_project/data")
    RESULTS_DIR = Path("../../23cs30011_project/results")
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    SAMPLE_PATH = DATA_DIR / SAMPLE_FILENAME
    INDEX_PATH = DATA_DIR / INDEX_FILENAME

GENERATION_RESULTS_PATH = RESULTS_DIR / GENERATION_RESULTS_FILENAME

# Loading the exact same 300-question sample and paragraph index used in Tasks 1 & 2
samples = [json.loads(l) for l in open(SAMPLE_PATH)]
passages = [json.loads(l) for l in open(INDEX_PATH)]
print(f"Loaded {len(samples)} questions, {len(passages)} paragraphs (same as Tasks 1 & 2).")

Loaded 300 questions, 7660 paragraphs (same as Tasks 1 & 2).


In [4]:
# Downloading the NLTK resources used below, skipping any that are already cached
import nltk

for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "stopwords"]:
    try:
        nltk.data.find(f"tokenizers/{pkg}")
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass
    try:
        nltk.data.find(f"taggers/{pkg}")
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass
    try:
        nltk.data.find(f"corpora/{pkg}")
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag
from nltk.corpus import stopwords

# Same stopword list and tokenizer as Task 1, rebuilt here so this notebook is self-contained
STOPWORDS = set(stopwords.words("english")) | {
    "who", "what", "when", "where", "which", "whom", "whose", "how", "many", "much"
}

TOKEN_RE = re.compile(r"[a-z0-9]+")


def tokenize(text):
    """Utility function to lowercase and tokenize text for BM25 / overlap scoring."""
    return TOKEN_RE.findall(text.lower())


# The EM / F1 / recall@k metric functions below are identical to Task 1's --
# reused as-is so Task 3's numbers stay directly comparable
def normalize_answer(s):
    """SQuAD-style normalization: lowercase, remove punctuation, remove articles, fix whitespace."""
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = " ".join(s.split())
    return s


def compute_em(pred, golds):
    """Utility function to return 1.0 if the prediction exactly matches any gold answer."""
    norm_pred = normalize_answer(pred)
    return float(any(norm_pred == normalize_answer(g) for g in golds))


def f1_score(pred, gold):
    """Utility function that computes token-level F1 between one prediction and one gold answer."""
    pred_tokens = normalize_answer(pred).split()
    gold_tokens = normalize_answer(gold).split()
    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def compute_f1(pred, golds):
    """Utility function that takes the best F1 across all acceptable gold answers."""
    return max(f1_score(pred, g) for g in golds) if golds else 0.0


def recall_at_k(hits, golds, k, passages_ref):
    """Utility function to check whether any gold answer string appears in the top-k retrieved passages."""
    norm_golds = [normalize_answer(g) for g in golds]
    for pid, _ in hits[:k]:
        norm_text = normalize_answer(passages_ref[pid]["text"])
        if any(g and g in norm_text for g in norm_golds):
            return 1.0
    return 0.0


print("Shared helpers ready.")

Shared helpers ready.


## 2. Rebuild BM25 ranking (Task 1's method)

In [5]:
# Rebuilding Task 1's BM25 index over the same paragraph pool
from rank_bm25 import BM25Okapi

corpus_texts = [p["text"] for p in passages]
tokenized_corpus = [tokenize(t) for t in tqdm(corpus_texts, desc="Tokenizing corpus")]
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

def bm25_retrieve(question, top_k=TOP_K_RETRIEVE):
    """Utility function that scores every passage against the question with BM25."""
    scores = bm25.get_scores(tokenize(question))
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_idx]

# Iterating through every question and retrieving its top-k BM25 passages
bm25_results = [bm25_retrieve(s["question"]) for s in tqdm(samples, desc="BM25 retrieval")]
print("BM25 index built over", len(tokenized_corpus), "paragraphs.")

Tokenizing corpus:   0%|          | 0/7660 [00:00<?, ?it/s]

BM25 retrieval:   0%|          | 0/300 [00:00<?, ?it/s]

BM25 index built over 7660 paragraphs.


## 3. Rebuild dense ranking (Task 2's method)

In [6]:
# Rebuilding Task 2's dense retriever: embed every passage once, then embed
# every question and search the FAISS index for its nearest passages
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer(EMBED_MODEL_NAME)

passage_embeddings = embedder.encode(
    corpus_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

# Building a flat inner-product FAISS index (embeddings are already normalized,
# so inner product is equivalent to cosine similarity)
dim = passage_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(passage_embeddings)

question_embeddings = embedder.encode(
    [s["question"] for s in samples], batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
).astype("float32")

# Searching the index for every question's top-k nearest passages
scores, indices = faiss_index.search(question_embeddings, TOP_K_RETRIEVE)
dense_results = [
    list(zip(indices[i].tolist(), scores[i].tolist())) for i in range(len(samples))
]
print("Dense index built over", faiss_index.ntotal, "paragraphs.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/120 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Dense index built over 7660 paragraphs.


## 4. Reciprocal Rank Fusion (constant=60)

In [7]:
def rrf_fuse(list_a, list_b, k=RRF_K, top_k=TOP_K_RETRIEVE):
    """Utility function that fuses two ranked (passage_id, score) lists via
    Reciprocal Rank Fusion: every passage gets 1/(k+rank) from each list it
    appears in, and the fused list is sorted by the summed score."""

    scores = {}
    for rank, (pid, _) in enumerate(list_a, start=1):
        scores[pid] = scores.get(pid, 0.0) + 1.0 / (k + rank)
    for rank, (pid, _) in enumerate(list_b, start=1):
        scores[pid] = scores.get(pid, 0.0) + 1.0 / (k + rank)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return ranked


# Fusing BM25 + dense rankings for every question
hybrid_results = [rrf_fuse(bm25_results[i], dense_results[i]) for i in range(len(samples))]

print("Example hybrid retrieval for:", samples[0]["question"])
for pid, sc in hybrid_results[0][:3]:
    print(f"  rrf={sc:.4f} doc={passages[pid]['doc_title']!r} text={passages[pid]['text'][:100]!r}")

Example hybrid retrieval for: Which actress was voted Miss Greenwich Village in 1942?
  rrf=0.0328 doc='Lauren Bacall' text='In 1941, Bacall studied for a year at the American Academy of Dramatic Arts in New York, where she m'
  rrf=0.0315 doc='Una Stubbs' text='Una Stubbs (1 May 1937 – 12 August 2021) was an English actress, television personality, and dancer '
  rrf=0.0161 doc='Buddy Holly' text='==== Richard Barone ==== In 2016, Richard Barone released his album Sorrows & Promises: Greenwich Vi'


## 5. recall@k: BM25 vs Dense vs Hybrid

In [8]:
def recall_table(results):
    """Utility function that averages recall@k over all questions for a given set of retrieval results."""

    rows = []
    for s, hits in zip(samples, results):
        rows.append({f"recall@{k}": recall_at_k(hits, s["gold_answers"], k, passages) for k in K_LIST})
    df = pd.DataFrame(rows)
    return {f"recall@{k}": df[f"recall@{k}"].mean() for k in K_LIST}


# Comparing BM25-only, dense-only, and the hybrid fusion side by side
retrieval_comparison = pd.DataFrame({
    "BM25 (Task 1)": recall_table(bm25_results),
    "Dense (Task 2)": recall_table(dense_results),
    "Hybrid RRF (Task 3)": recall_table(hybrid_results),
})
print(retrieval_comparison)

           BM25 (Task 1)  Dense (Task 2)  Hybrid RRF (Task 3)
recall@1        0.533333        0.513333             0.546667
recall@3        0.660000        0.620000             0.663333
recall@5        0.716667        0.670000             0.713333
recall@10       0.743333        0.730000             0.780000


## 6. Generation: hybrid top-5 -> `Qwen2.5-7B-Instruct` (4-bit), greedy decode; retry ungrounded with top-10

In [9]:
def groundedness_overlap(pred, hits, top_k):
    """Utility function that scores how much of the answer's content-word vocabulary
    actually appears in the retrieved evidence text -- this is the groundedness signal."""

    pred_tokens = set(tokenize(pred)) - STOPWORDS
    if not pred_tokens:
        return 0.0
    evidence_text = " ".join(passages[pid]["text"] for pid, _ in hits[:top_k])
    evidence_tokens = set(tokenize(evidence_text)) - STOPWORDS
    return len(pred_tokens & evidence_tokens) / len(pred_tokens)


# On Colab: load the 4-bit generator once, then run pass 1 (top-5) generation
# followed by pass 2 (top-10 retry) on whatever pass 1 flagged as ungrounded.
# Locally: just reuse a previous Colab run's saved generations file.
if IN_COLAB:
    import time
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    torch.manual_seed(SEED)

    # Loading the generator in 4-bit (same model/settings as Task 2)
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

    print("Loading tokenizer/model:", GEN_MODEL_NAME)
    gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
    gen_model = AutoModelForCausalLM.from_pretrained(
        GEN_MODEL_NAME, quantization_config=quant_config, device_map="auto",
    )
    gen_model.eval()

    def truncate_to_tokens(text, max_tokens):
        """Utility function that trims a passage down to at most max_tokens generator tokens."""
        ids = gen_tokenizer.encode(text, add_special_tokens=False)
        if len(ids) <= max_tokens:
            return text
        return gen_tokenizer.decode(ids[:max_tokens], skip_special_tokens=True)

    def build_prompt(question, top_passages, top_k, retry=False):
        """Utility function that builds the chat prompt for a question: the normal
        prompt for pass 1, or a stricter re-read prompt (wider context, allowed to
        say UNKNOWN) when this is the groundedness retry."""

        passage_block = "\n\n".join(
            f"Passage {i+1} ({passages[pid]['doc_title']}): "
            f"{truncate_to_tokens(passages[pid]['text'], MAX_PASSAGE_TOKENS)}"
            for i, (pid, _) in enumerate(top_passages[:top_k])
        )
        if retry:
            user_msg = (
                "Your previous answer did not clearly match the retrieved evidence. Here is a "
                "WIDER set of passages. Re-read them carefully. Answer using ONLY information "
                "explicitly stated in the passages below. Respond with just the short answer, "
                "no explanation. If the answer is genuinely not present in these passages, "
                "respond exactly: UNKNOWN\n\n"
                f"{passage_block}\n\nQuestion: {question}\nAnswer:"
            )
        else:
            user_msg = (
                "Answer the question using ONLY the information in the passages below. "
                "Respond with just the short answer, no explanation, no full sentence.\n\n"
                f"{passage_block}\n\nQuestion: {question}\nAnswer:"
            )
        messages = [
            {"role": "system", "content": "You are a concise, factual question-answering assistant."},
            {"role": "user", "content": user_msg},
        ]
        return gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def generate_answer(question, top_passages, top_k, retry=False):
        """Utility function that builds the prompt and greedily decodes the generator's answer."""

        prompt = build_prompt(question, top_passages, top_k, retry=retry)
        inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
        with torch.no_grad():
            out = gen_model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                pad_token_id=gen_tokenizer.eos_token_id,
            )
        new_tokens = out[0][inputs["input_ids"].shape[1]:]
        return gen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Pass 1: generate with top-5, same settings as Task 2.
    generations = []
    t0 = time.time()
    for i, (s, hits) in enumerate(zip(samples, hybrid_results)):
        pred = generate_answer(s["question"], hits, top_k=TOP_K)
        overlap = groundedness_overlap(pred, hits, top_k=TOP_K)
        generations.append({
            "question_id": s["question_id"], "question": s["question"],
            "gold_answers": s["gold_answers"], "prediction_raw": pred,
            "overlap": overlap, "is_grounded": overlap >= GROUNDEDNESS_THRESHOLD,
        })
        if (i + 1) % 10 == 0:
            elapsed = time.time() - t0
            print(f"Pass 1: {i+1}/{len(samples)} done, {elapsed:.0f}s elapsed")

    # Counting how many pass-1 answers got flagged as ungrounded, before we retry them
    n_ungrounded = sum(1 for g in generations if not g["is_grounded"])
    print(f"\nPass 1 done. {n_ungrounded}/{len(samples)} flagged ungrounded -> retrying with top-10.")

    # Pass 2: retry only the flagged-ungrounded questions with expanded top-10 context.
    t0 = time.time()
    retried = 0
    for i, (s, hits, g) in enumerate(zip(samples, hybrid_results, generations)):
        if g["is_grounded"]:
            g["prediction_retry"] = g["prediction_raw"]  # unchanged
            continue
        retry_pred = generate_answer(s["question"], hits, top_k=TOP_K_RETRIEVE, retry=True)
        g["prediction_retry"] = retry_pred
        retried += 1
        if retried % 10 == 0:
            elapsed = time.time() - t0
            print(f"Pass 2 (retry): {retried}/{n_ungrounded} done, {elapsed:.0f}s elapsed")

    print(f"\nPass 2 done. {retried} questions retried.")

    # Persisting both passes' predictions so this notebook can be re-scored locally later
    with open(GENERATION_RESULTS_PATH, "w") as f:
        for r in generations:
            f.write(json.dumps(r) + "\n")
    print("Saved", GENERATION_RESULTS_PATH)
    print("\nDownload this file and place it in this repo's results/ folder:")
    files.download(str(GENERATION_RESULTS_PATH))

else:
    # Local run: no GPU, so just load a previously-downloaded generations file
    assert GENERATION_RESULTS_PATH.exists(), (
        f"{GENERATION_RESULTS_PATH} not found. Run this notebook on Colab (GPU) first to "
        "generate it, or place a previous run's output there."
    )
    generations = [json.loads(l) for l in open(GENERATION_RESULTS_PATH)]
    print(f"Local run: loaded {len(generations)} pre-generated predictions from {GENERATION_RESULTS_PATH}.")

# Indexing the generations by question_id so later cells can look them up directly
gen_by_qid = {g["question_id"]: g for g in generations}
assert set(gen_by_qid) == {s["question_id"] for s in samples}, "generation set doesn't match the sample set"
print("Question IDs match the fixed sample exactly.")

Local run: loaded 300 pre-generated predictions from ../../results/task3_generation_results.jsonl.
Question IDs match the fixed sample exactly.


## 7. Score hybrid generations (pre-groundedness-fallback): EM, F1

In [10]:
# Iterating through every question and building one results record per question:
# gold answers, pass-1 prediction, groundedness flag, and EM/F1 for pass-1
records = []
for s, hits in zip(samples, hybrid_results):
    g = gen_by_qid[s["question_id"]]
    pred = g["prediction_raw"]
    golds = s["gold_answers"]
    records.append({
        "question_id": s["question_id"],
        "question": s["question"],
        "gold_answers": golds,
        "prediction_raw": pred,
        "prediction_retry": g["prediction_retry"],
        "overlap": g["overlap"],
        "is_grounded": g["is_grounded"],
        "em_raw": compute_em(pred, golds),
        "f1_raw": compute_f1(pred, golds),
        "top1_doc": passages[hits[0][0]]["doc_title"],
        "hybrid_hits": hits,
    })

task3_df = pd.DataFrame(records)

# Headline Task 3 metrics are the pass-1 (no-retry) numbers
raw_metrics = {"EM": task3_df["em_raw"].mean(), "F1": task3_df["f1_raw"].mean()}
for k in K_LIST:
    raw_metrics[f"recall@{k}"] = retrieval_comparison["Hybrid RRF (Task 3)"][f"recall@{k}"]

# Pulling in Task 1 & Task 2's saved metrics (if available) for a side-by-side comparison
task1_metrics_path = Path("../../results/task1_metrics.json")
task2_metrics_path = Path("../../results/task2_metrics.json")
compare_cols = {"Task 3 hybrid (HEADLINE, pass-1)": raw_metrics}
if task1_metrics_path.exists():
    compare_cols["Task 1 (BM25 + heuristic)"] = json.load(open(task1_metrics_path))
if task2_metrics_path.exists():
    compare_cols["Task 2 (dense + LLM)"] = json.load(open(task2_metrics_path))

print(pd.DataFrame(compare_cols))

             Task 3 hybrid (HEADLINE, pass-1)  Task 1 (BM25 + heuristic)  \
EM                                   0.670000                   0.020000   
F1                                   0.711073                   0.092664   
recall@1                             0.546667                   0.533333   
recall@3                             0.663333                   0.660000   
recall@5                             0.713333                   0.716667   
recall@10                            0.780000                   0.743333   
n_questions                               NaN                 300.000000   

             Task 2 (dense + LLM)  
EM                       0.660000  
F1                       0.697888  
recall@1                 0.513333  
recall@3                 0.620000  
recall@5                 0.670000  
recall@10                0.730000  
n_questions            300.000000  


## 8. Groundedness check: lexical overlap between answer and retrieved evidence

In [11]:
# Checking whether the groundedness flag actually tracks correctness:
# correct answers should mostly be flagged grounded, incorrect ones mostly ungrounded
correct = task3_df["em_raw"] == 1.0
incorrect = ~correct
frac_correct_grounded = task3_df.loc[correct, "is_grounded"].mean()
frac_incorrect_ungrounded = (~task3_df.loc[incorrect, "is_grounded"]).mean()

print(f"Correct answers with high overlap (grounded):   {frac_correct_grounded:.3f} "
      f"({correct.sum()} correct total)")
print(f"Incorrect answers with low overlap (ungrounded): {frac_incorrect_ungrounded:.3f} "
      f"({incorrect.sum()} incorrect total)")
print(f"Groundedness-check accuracy "
      f"(is_grounded predicts correctness): "
      f"{(task3_df['is_grounded'] == correct).mean():.3f}")

Correct answers with high overlap (grounded):   0.915 (201 correct total)
Incorrect answers with low overlap (ungrounded): 0.485 (99 incorrect total)
Groundedness-check accuracy (is_grounded predicts correctness): 0.773


## 9. Experiment: does retrying ungrounded answers (top-10 + "say UNKNOWN") help?

In [12]:
# For each question, keeping the pass-1 answer if it was grounded, otherwise
# swapping in the retry (top-10) answer -- this is the "with retry" variant,
# reported only as a reference/diagnostic, never as the headline result
final_predictions = []
retry_used = []
for row in task3_df.itertuples():
    if row.is_grounded:
        final_predictions.append(row.prediction_raw)
        retry_used.append(False)
    else:
        final_predictions.append(row.prediction_retry)
        retry_used.append(True)

task3_df["prediction_final"] = final_predictions
task3_df["retry_used"] = retry_used
task3_df["em_final"] = [compute_em(p, g) for p, g in zip(final_predictions, task3_df["gold_answers"])]
task3_df["f1_final"] = [compute_f1(p, g) for p, g in zip(final_predictions, task3_df["gold_answers"])]

print(f"Retry used on {sum(retry_used)}/{len(task3_df)} questions "
      f"({sum(retry_used)/len(task3_df):.1%}).")

Retry used on 65/300 questions (21.7%).


## 10. Final comparison: Task 1 vs Task 2 vs Task 3 (headline = pass-1; retry shown for reference)

In [13]:
# Saving the full per-question results, and the pass-1 / with-retry / groundedness
# summary metrics (pass1_HEADLINE is the number actually reported for Task 3)
task3_df.to_csv(RESULTS_DIR / "task3_per_question_results.csv", index=False)

final_metrics = {
    "n_questions": len(task3_df),
    "EM": task3_df["em_final"].mean(),
    "F1": task3_df["f1_final"].mean(),
}
for k in K_LIST:
    final_metrics[f"recall@{k}"] = retrieval_comparison["Hybrid RRF (Task 3)"][f"recall@{k}"]

groundedness_metrics = {
    "frac_correct_grounded": float(frac_correct_grounded),
    "frac_incorrect_ungrounded": float(frac_incorrect_ungrounded),
    "groundedness_check_accuracy": float((task3_df["is_grounded"] == correct).mean()),
    "retry_rate": float(sum(retry_used) / len(task3_df)),
}

with open(RESULTS_DIR / "task3_metrics.json", "w") as f:
    json.dump({"pass1_HEADLINE": raw_metrics, "with_retry_reference_only": final_metrics,
               "groundedness": groundedness_metrics}, f, indent=2)

compare_cols["Task 3 retry experiment (reference, net negative)"] = final_metrics
print(pd.DataFrame(compare_cols))
print("\nGroundedness:", json.dumps(groundedness_metrics, indent=2))
print(f"\nRetry net effect: {sum((task3_df['retry_used']) & (task3_df['em_raw']==0) & (task3_df['em_final']==1))} fixed, "
      f"{sum((task3_df['retry_used']) & (task3_df['em_raw']==1) & (task3_df['em_final']==0))} broken "
      f"-> headline Task 3 result uses PASS-1 (no retry), not this column.")

# Checking the retry's net effect directly: how many it fixed vs. how many it broke
if IN_COLAB:
    files.download(str(RESULTS_DIR / "task3_per_question_results.csv"))
    files.download(str(RESULTS_DIR / "task3_metrics.json"))

             Task 3 hybrid (HEADLINE, pass-1)  Task 1 (BM25 + heuristic)  \
EM                                   0.670000                   0.020000   
F1                                   0.711073                   0.092664   
recall@1                             0.546667                   0.533333   
recall@3                             0.663333                   0.660000   
recall@5                             0.713333                   0.716667   
recall@10                            0.780000                   0.743333   
n_questions                               NaN                 300.000000   

             Task 2 (dense + LLM)  \
EM                       0.660000   
F1                       0.697888   
recall@1                 0.513333   
recall@3                 0.620000   
recall@5                 0.670000   
recall@10                0.730000   
n_questions            300.000000   

             Task 3 retry experiment (reference, net negative)  
EM                           

## 11. Qualitative analysis: retry corrections and misses

In [14]:
def show_retry_examples(df, n=5, outcome="fixed", seed=SEED):
    """Utility function to print n random examples of a given retry outcome
    (fixed / broke / still wrong), for qualitative inspection."""

    fb = df[df["retry_used"]]
    if outcome == "fixed":
        subset = fb[(fb["em_raw"] == 0) & (fb["em_final"] == 1)]
    elif outcome == "broke":
        subset = fb[(fb["em_raw"] == 1) & (fb["em_final"] == 0)]
    else:
        subset = fb[(fb["em_raw"] == 0) & (fb["em_final"] == 0)]
    subset = subset.sample(min(n, len(subset)), random_state=seed)
    for row in subset.itertuples():
        print(f"Q: {row.question}")
        print(f"   Gold: {row.gold_answers}")
        print(f"   Pass-1 (top-5, overlap={row.overlap:.2f}): {row.prediction_raw!r}")
        print(f"   Retry (top-10): {row.prediction_retry!r}")
        print("-" * 80)


# Showing a few cases where the retry helped, and a few where it hurt
print(f">>> Retry FIXED an error ({((task3_df['retry_used']) & (task3_df['em_raw']==0) & (task3_df['em_final']==1)).sum()} cases)")
show_retry_examples(task3_df, outcome="fixed")

print(f"\n>>> Retry BROKE a correct answer ({((task3_df['retry_used']) & (task3_df['em_raw']==1) & (task3_df['em_final']==0)).sum()} cases)")
show_retry_examples(task3_df, outcome="broke")

>>> Retry FIXED an error (1 cases)
Q: This quiz is printed on A4 paper which has a length of 297mm. How wide, in millimetres, is A4 paper?
   Gold: ['210', 'two hundred and ten']
   Pass-1 (top-5, overlap=0.00): '210mm'
   Retry (top-10): '210'
--------------------------------------------------------------------------------

>>> Retry BROKE a correct answer (14 cases)
Q: Which Shakespeare play centres around a novice nun called 'Isabella' whosebrother is to be executed unless she succumbs to the attentions of 'Angelo'?
   Gold: ["'MEASURE FOR MEASURE'", 'Keepdown', 'Over done', 'Keep down', 'Kate Keepdown', 'Overdone', 'Measure for Measure', 'Mistress Overdone', 'Measure For Measure', 'Abhorson', 'Measure for measure', 'measure for measure', 'over done', 'keepdown', 'abhorson', 'kate keepdown', 'overdone', 'mistress overdone', 'keep down']
   Pass-1 (top-5, overlap=0.00): 'Measure for Measure'
   Retry (top-10): 'UNKNOWN'
----------------------------------------------------------------

## 12. Summary

Headline (pass-1, n=300): EM=0.67, F1=0.711, recall@1=0.547, recall@5=0.713, recall@10=0.78 --
improves on both Task 1 and Task 2 alone. Groundedness check: 91.5% of correct answers flagged
grounded, 48.5% of incorrect answers flagged ungrounded (accuracy=0.773). Retrying ungrounded
answers with expanded context does NOT improve EM (1 fixed, 14 broken, net -13/300) -- kept as a
documented negative result, not used in the headline. Full discussion: see report.